# Оценка моделей

## Импорты

In [1]:
from pathlib import Path
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bert_score import score

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Настройки

In [2]:
def find_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data").exists() and (folder / "reports").exists():
            return folder
    raise FileNotFoundError("Корень проекта не найден")


def style_plot(fig, title, subtitle, height=500):
    fig.update_layout(
        title={"text": f"{title}<br><sup>{subtitle}</sup>", "x": 0.02, "xanchor": "left"},
        template="plotly_white",
        height=height,
        font={"family": "Segoe UI", "size": 13, "color": "#1f2937"},
        margin={"l": 30, "r": 30, "t": 90, "b": 40},
        legend_title_text="",
    )
    fig.update_xaxes(gridcolor="#e5e7eb", zerolinecolor="#6b7280")
    fig.update_yaxes(gridcolor="#e5e7eb")
    return fig


ROOT = find_root()
REPORT_DIR = ROOT / "reports" / "evaluation"
RESULT_DIR = REPORT_DIR / "all_models"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
BERT_BATCH_SIZE = 16
BERT_DEVICE = None
BLUE = "#2563eb"
GREEN = "#0bf53e"
PINK = "#db2777"
LIGHT_BLUE = "#93c5fd"

## GQA-ru

In [3]:
gqa_files = [
    ("Qwen2.5-VL базовая", "qwen_lora_v3_predictions.csv", "baseline_answer"),
    ("Qwen2.5-VL + LoRA #1", "qwen_lora_predictions.csv", "lora_answer"),
    ("Qwen2.5-VL + LoRA #2", "qwen_lora_v2_predictions.csv", "lora_answer"),
    ("Qwen2.5-VL + LoRA #3", "qwen_lora_v3_predictions.csv", "lora_answer"),
    ("SmolVLM базовая", "smolvlm_lora_predictions.csv", "baseline_answer"),
    ("SmolVLM + LoRA", "smolvlm_lora_predictions.csv", "lora_answer"),
]

def normalize(text):
    text = str(text).lower().replace("ё", "е").strip()
    text = re.sub(r"[^a-zа-я0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    answers = {"да": "yes", "нет": "no", "yes": "yes", "no": "no"}
    return answers.get(text, text)

gqa_data = ROOT / "data" / "GQA-ru" / "testdev_balanced_instructions" / "testdev-00000-of-00001.parquet"
gqa_types = pd.read_parquet(gqa_data, columns=["id", "types"])
gqa_types["id"] = gqa_types["id"].astype(str)
if gqa_types["id"].duplicated().any():
    raise ValueError("В исходном testdev GQA-ru обнаружены повторяющиеся ID")
gqa_expected_ids = set(gqa_types["id"])
gqa_expected_examples = len(gqa_expected_ids)

gqa_parts = []
for model_name, file_name, prediction_column in gqa_files:
    source_part = pd.read_csv(REPORT_DIR / file_name)
    source_part["id"] = source_part["id"].astype(str)
    if source_part["id"].duplicated().any():
        raise ValueError(f"В {file_name} есть повторяющиеся ID")
    current_ids = set(source_part["id"])
    if current_ids != gqa_expected_ids:
        missing = len(gqa_expected_ids - current_ids)
        extra = len(current_ids - gqa_expected_ids)
        raise ValueError(f"{file_name}: неполное покрытие GQA-ru (пропущено {missing}, лишних {extra})")
    error_column = f"{prediction_column}_error"
    if error_column in source_part and source_part[error_column].notna().any():
        raise ValueError(f"{file_name}: есть ошибки инференса в {error_column}")
    part = source_part
    part = part[["id", "question", "true_answer", prediction_column]].copy()
    part.columns = ["id", "question", "reference", "prediction"]
    part["model"] = model_name
    gqa_parts.append(part)

gqa = pd.concat(gqa_parts, ignore_index=True)
gqa["id"] = gqa["id"].astype(str)
gqa["reference"] = gqa["reference"].fillna("").astype(str)
gqa["prediction"] = gqa["prediction"].fillna("").astype(str)
gqa["exact_match"] = gqa.apply(lambda row: normalize(row["reference"]) == normalize(row["prediction"]), axis=1)

gqa_types["category"] = gqa_types["types"].apply(lambda value: value.get("semantic", "другое"))
gqa = gqa.merge(gqa_types[["id", "category"]], on="id", how="left", validate="many_to_one")
gqa["category"] = gqa["category"].fillna("другое")
gqa_coverage = (
    gqa.groupby("model", as_index=False)
    .agg(rows=("id", "size"), unique_examples=("id", "nunique"))
)
gqa_coverage["expected_examples"] = gqa_expected_examples
gqa_coverage["full_coverage"] = (
    gqa_coverage["rows"].eq(gqa_expected_examples)
    & gqa_coverage["unique_examples"].eq(gqa_expected_examples)
)
if not gqa_coverage["full_coverage"].all():
    raise ValueError("Не все модели проверены на полном testdev GQA-ru")
display(gqa_coverage)
gqa.head()

,model,rows,unique_examples,expected_examples,full_coverage
0,Qwen2.5-VL + LoRA #1,12216,12216,12216,True
1,Qwen2.5-VL + LoRA #2,12216,12216,12216,True
2,Qwen2.5-VL + LoRA #3,12216,12216,12216,True
3,Qwen2.5-VL базовая,12216,12216,12216,True
4,SmolVLM + LoRA,12216,12216,12216,True
5,SmolVLM базовая,12216,12216,12216,True


,id,question,reference,prediction,model,exact_match,category
0,201307251,Пасмурно?,нет,no,Qwen2.5-VL базовая,True,global
1,201640614,Кто носит платье?,женщины,woman,Qwen2.5-VL базовая,False,rel
2,202225914,Посуда на столе выглядит чистой и черной?,нет,no,Qwen2.5-VL базовая,True,attr
3,2062325,Носит ли мокрый на вид серфер гидрокостюм?,да,yes,Qwen2.5-VL базовая,True,rel
4,201303229,Какой высоты стул внизу фотографии?,низкий,10 feet,Qwen2.5-VL базовая,False,attr


## Exact Match

In [4]:
gqa_exact = (
    gqa.groupby("model", as_index=False)
    .agg(exact_match=("exact_match", "mean"), examples=("exact_match", "size"))
    .sort_values("exact_match", ascending=False)
)
gqa_exact

,model,exact_match,examples
1,Qwen2.5-VL + LoRA #2,0.537246,12216
2,Qwen2.5-VL + LoRA #3,0.535200,12216
0,Qwen2.5-VL + LoRA #1,0.500573,12216
3,Qwen2.5-VL базовая,0.319826,12216
4,SmolVLM + LoRA,0.291830,12216
5,SmolVLM базовая,0.202849,12216


In [5]:
plot_data = gqa_exact.sort_values("exact_match")
fig = px.bar(plot_data, x="exact_match", y="model", orientation="h", text=plot_data["exact_match"].map(lambda x: f"{x:.1%}"), color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", marker_line_color="#1e3a8a", marker_line_width=0.8)
fig.update_xaxes(range=[0, 0.65], tickformat=".0%", title="Exact Match")
fig.update_yaxes(title="")
gqa_examples_label = f"{gqa_expected_examples:,}".replace(",", " ")
style_plot(fig, "GQA-ru: Exact Match", f"Все тестовые данные: по {gqa_examples_label} примеров для каждой модели").show()

### Вывод по Exact Match

- Метрика рассчитана на полном `testdev` GQA-ru: **12 216 примеров для каждой модели**.
- Лучший Exact Match у `Qwen2.5-VL + LoRA #2` — **53,72%**. LoRA #3 практически не отстаёт с **53,52%**: разница составляет только **0,20 п.п.**
- LoRA #1 получила **50,06%**, а базовая Qwen — **31,98%**. Прирост трёх адаптеров относительно базы равен **18,07**, **21,74** и **21,54 п.п.** соответственно.
- У SmolVLM адаптация повысила Exact Match с **20,28%** до **29,18%**, однако обе версии остаются слабее моделей семейства Qwen.

## BERTScore

In [6]:
BERT_EMPTY_PLACEHOLDER = "empty prediction"


def prepare_bertscore_texts(series):
    """Заменяет пустые строки: bert-score 0.3.13 несовместим с ними в transformers 5."""
    texts = series.fillna("").astype(str).str.strip()
    return texts.mask(texts.eq(""), BERT_EMPTY_PLACEHOLDER).tolist()


bert_candidates = prepare_bertscore_texts(gqa["prediction"])
bert_references = prepare_bertscore_texts(gqa["reference"])
empty_prediction_mask = gqa["prediction"].fillna("").astype(str).str.strip().eq("")
empty_reference_mask = gqa["reference"].fillna("").astype(str).str.strip().eq("")
empty_bert_mask = empty_prediction_mask | empty_reference_mask
empty_predictions = int(empty_prediction_mask.sum())
print(f"Пустых предсказаний заменено перед BERTScore: {empty_predictions}")

bert_precision, bert_recall, bert_f1 = score(
    bert_candidates,
    bert_references,
    lang="ru",
    batch_size=BERT_BATCH_SIZE,
    device=BERT_DEVICE,
    verbose=True,
)
gqa["bertscore_precision"] = bert_precision.cpu().numpy()
gqa["bertscore_recall"] = bert_recall.cpu().numpy()
gqa["bertscore_f1"] = bert_f1.cpu().numpy()
bert_columns = ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
gqa.loc[empty_bert_mask, bert_columns] = 0.0

gqa_bert = (
    gqa.groupby("model", as_index=False)
    .agg(
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
    )
    .sort_values("bertscore_f1", ascending=False)
)
gqa_bert

Пустых предсказаний заменено перед BERTScore: 2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11706.24it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 399/399 [00:02<00:00, 137.19it/s]


computing greedy matching.


100%|██████████| 4581/4581 [00:14<00:00, 314.86it/s]

done in 17.47 seconds, 4194.95 sentences/sec


,model,bertscore_precision,bertscore_recall,bertscore_f1
1,Qwen2.5-VL + LoRA #2,0.908978,0.904446,0.906516
2,Qwen2.5-VL + LoRA #3,0.907888,0.903537,0.905526
0,Qwen2.5-VL + LoRA #1,0.900475,0.896211,0.898145
4,SmolVLM + LoRA,0.853854,0.848809,0.851108
3,Qwen2.5-VL базовая,0.810846,0.802432,0.806401
5,SmolVLM базовая,0.689655,0.669268,0.679032


In [7]:
bert_plot = gqa_bert.sort_values("bertscore_f1")
fig = go.Figure()
fig.add_bar(name="Precision", y=bert_plot["model"], x=bert_plot["bertscore_precision"], orientation="h", marker_color=LIGHT_BLUE)
fig.add_bar(name="Recall", y=bert_plot["model"], x=bert_plot["bertscore_recall"], orientation="h", marker_color=GREEN)
fig.add_bar(name="F1", y=bert_plot["model"], x=bert_plot["bertscore_f1"], orientation="h", marker_color=BLUE)
fig.update_layout(barmode="group")
fig.update_xaxes(range=[0.6, 0.95], tickformat=".0%", title="BERTScore")
fig.update_yaxes(title="")
style_plot(fig, "GQA-ru: BERTScore", f"Семантическая близость; по {gqa_examples_label} тестовых примеров", 560).show()

### Вывод по BERTScore

- На всех 12 216 примерах лучший BERTScore F1 также у `Qwen2.5-VL + LoRA #2` — **0,907**.
- LoRA #3 получила **0,906**, а LoRA #1 — **0,898**. Разница между второй и третьей версиями составляет около **0,001**.
- У базовой Qwen F1 равен **0,806**, поэтому каждый из трёх адаптеров заметно улучшает смысловую близость ответов.
- SmolVLM с LoRA получила **0,851** против **0,679** у базовой модели.
- BERTScore заметно выше Exact Match: часть ответов близка по смыслу, но отличается формой слова, числом или формулировкой.

## Accuracy по категориям GQA-ru

In [8]:
gqa_categories = (
    gqa.groupby(["model", "category"], as_index=False)
    .agg(accuracy=("exact_match", "mean"), examples=("exact_match", "size"))
    .sort_values(["model", "accuracy"])
)
gqa_categories

,model,category,accuracy,examples
1,Qwen2.5-VL + LoRA #1,cat,0.368610,1115
4,Qwen2.5-VL + LoRA #1,rel,0.394253,5116
2,Qwen2.5-VL + LoRA #1,global,0.439490,157
0,Qwen2.5-VL + LoRA #1,attr,0.596127,5061
3,Qwen2.5-VL + LoRA #1,obj,0.783572,767
6,Qwen2.5-VL + LoRA #2,cat,0.420628,1115
9,Qwen2.5-VL + LoRA #2,rel,0.430414,5116
7,Qwen2.5-VL + LoRA #2,global,0.464968,157
5,Qwen2.5-VL + LoRA #2,attr,0.628730,5061
8,Qwen2.5-VL + LoRA #2,obj,0.830508,767


In [9]:
matrix = gqa_categories.pivot(index="model", columns="category", values="accuracy")
fig = px.imshow(matrix, text_auto=".1%", aspect="auto", zmin=0, zmax=1, color_continuous_scale=[[0, "#eff6ff"], [0.5, "#60a5fa"], [1, "#1e3a8a"]], labels={"color": "Accuracy"})
fig.update_xaxes(title="Категория")
fig.update_yaxes(title="")
global_examples = int(gqa_categories.loc[gqa_categories["category"].eq("global"), "examples"].max())
style_plot(fig, "GQA-ru: качество по категориям", f"Exact Match на полном testdev; global содержит {global_examples} примеров", 600).show()

### Вывод по категориям GQA-ru

- Qwen LoRA #2 лидирует по общей метрике и особенно уверенно отвечает на вопросы об объектах — **83,1%** и атрибутах — **62,9%**.
- LoRA #3 очень близка: **82,8%** на объектах и **62,2%** на атрибутах. При этом она немного лучше LoRA #2 в отношениях — **43,2%** против **43,0%**, и категориях объектов — **42,2%** против **42,1%**.
- LoRA #1 получила **78,4%** на объектах, **59,6%** на атрибутах, **39,4%** на отношениях и **36,9%** на категориях объектов.
- `global` — самая маленькая категория: в ней **157 примеров**, тогда как в `rel` и `attr` больше пяти тысяч. Поэтому её значения менее устойчивы, чем результаты крупных категорий.
- LoRA улучшила SmolVLM во всех пяти категориях, но по Exact Match эта модель всё ещё заметно уступает Qwen.

## MMBench-ru

In [10]:
MMBENCH_MODEL_ORDER = [
    "Qwen2.5-VL базовая",
    "Qwen2.5-VL + LoRA #1",
    "Qwen2.5-VL + LoRA #2",
    "Qwen2.5-VL + LoRA #3",
    "SmolVLM базовая",
    "SmolVLM + LoRA",
]

mmbench = pd.read_csv(REPORT_DIR / "mmbench_ru_predictions.csv")
mmbench_source = pd.read_parquet(
    ROOT / "data" / "MMBench-ru" / "mmbench_ru_dev.parquet", columns=["index"]
)
if mmbench_source["index"].duplicated().any():
    raise ValueError("В исходном MMBench-ru обнаружены повторяющиеся index")
mmbench_expected_examples = len(mmbench_source)
mmbench_expected_row_ids = set(range(mmbench_expected_examples))
available_models = set(mmbench["model_name"].unique())
missing_models = sorted(set(MMBENCH_MODEL_ORDER) - available_models)
unexpected_models = sorted(available_models - set(MMBENCH_MODEL_ORDER))
if missing_models or unexpected_models:
    raise ValueError(
        f"Неверный набор моделей MMBench-ru: отсутствуют {missing_models}, лишние {unexpected_models}"
    )

mmbench_coverage_rows = []
for model_name, part in mmbench.groupby("model_name"):
    duplicate_rows = int(part["row_id"].duplicated().sum())
    current_row_ids = set(part["row_id"])
    missing_rows = len(mmbench_expected_row_ids - current_row_ids)
    extra_rows = len(current_row_ids - mmbench_expected_row_ids)
    inference_errors = int(part["error"].notna().sum())
    full_coverage = not (duplicate_rows or missing_rows or extra_rows or inference_errors)
    mmbench_coverage_rows.append({
        "model_name": model_name,
        "rows": len(part),
        "unique_examples": part["row_id"].nunique(),
        "expected_examples": mmbench_expected_examples,
        "errors": inference_errors,
        "full_coverage": full_coverage,
    })
mmbench_coverage = pd.DataFrame(mmbench_coverage_rows)
if not mmbench_coverage["full_coverage"].all():
    raise ValueError("Не все модели проверены на полном dev MMBench-ru")

mmbench = mmbench[mmbench["error"].isna()].copy()
mmbench["is_correct"] = mmbench["is_correct"].astype(bool)
mmbench_metrics = (
    mmbench.groupby("model_name", as_index=False)
    .agg(accuracy=("is_correct", "mean"), examples=("is_correct", "size"))
    .sort_values("accuracy", ascending=False)
)
display(mmbench_coverage)
display(mmbench.head())
display(mmbench_metrics)

,model_name,rows,unique_examples,expected_examples,errors,full_coverage
0,Qwen2.5-VL + LoRA #1,3910,3910,3910,0,True
1,Qwen2.5-VL + LoRA #2,3910,3910,3910,0,True
2,Qwen2.5-VL + LoRA #3,3910,3910,3910,0,True
3,Qwen2.5-VL базовая,3910,3910,3910,0,True
4,SmolVLM + LoRA,3910,3910,3910,0,True
5,SmolVLM базовая,3910,3910,3910,0,True


,model_name,family,row_id,parquet_row_id,split,category,task_type,metric_name,question,hint,options,true_answer,true_answer_norm,prediction,prediction_norm,is_correct,inference_time_s,error
0,Qwen2.5-VL базовая,qwen,0,0,dev,physical_property_reasoning,multiple_choice,accuracy,Какая часть яблони может вырасти в новое дерево?,На этой диаграмме показан жизненный цикл яблони.,A. семя\nB. лист,A,A,A,A,True,0.235373,NaN
1,Qwen2.5-VL базовая,qwen,1,1,dev,function_reasoning,multiple_choice,accuracy,Из какого материала сделана эта лопатка?,NaN,A. резина\nB. хлопок,A,A,A,A,True,0.197503,NaN
2,Qwen2.5-VL базовая,qwen,2,2,dev,image_scene,multiple_choice,accuracy,Это место многолюдное?,NaN,A. да\nB. нет,A,A,A,A,True,0.215621,NaN
3,Qwen2.5-VL базовая,qwen,3,3,dev,image_quality,multiple_choice,accuracy,Какое изображение более красочное?,NaN,A. Первое изображение\nB. Второе изображение,A,A,A,A,True,0.209532,NaN
4,Qwen2.5-VL базовая,qwen,4,4,dev,image_quality,multiple_choice,accuracy,Какая картинка ярче?,NaN,A. Первая картинка\nB. Вторая картинка,A,A,A,A,True,0.213673,NaN


,model_name,accuracy,examples
2,Qwen2.5-VL + LoRA #3,0.806138,3910
1,Qwen2.5-VL + LoRA #2,0.803325,3910
3,Qwen2.5-VL базовая,0.799233,3910
0,Qwen2.5-VL + LoRA #1,0.797954,3910
5,SmolVLM базовая,0.373402,3910
4,SmolVLM + LoRA,0.365473,3910


In [11]:
plot_data = mmbench_metrics.sort_values("accuracy")
fig = px.bar(plot_data, x="accuracy", y="model_name", orientation="h", text=plot_data["accuracy"].map(lambda x: f"{x:.1%}"), color_discrete_sequence=[GREEN])
fig.update_traces(textposition="outside", marker_line_color="#92400e", marker_line_width=0.8)
fig.update_xaxes(range=[0, 0.88], tickformat=".0%", title="Accuracy")
fig.update_yaxes(title="")
mmbench_examples = int(mmbench_metrics["examples"].min())
mmbench_examples_label = f"{mmbench_examples:,}".replace(",", " ")
style_plot(fig, "MMBench-ru: Accuracy", f"Все тестовые данные: по {mmbench_examples_label} примеров для каждой модели").show()

### Вывод по MMBench-ru

- Лучший результат на полном наборе показала `Qwen2.5-VL + LoRA #3` — **80,61% Accuracy**.
- LoRA #2 набрала **80,33%**, базовая Qwen — **79,92%**, а LoRA #1 — **79,80%**. Разница между четырьмя версиями Qwen меньше одного процентного пункта.
- Относительно базовой Qwen третья LoRA прибавила **0,69 п.п.**, вторая — **0,41 п.п.**, первая уступила базе **0,13 п.п.**
- Базовая SmolVLM получила **37,34%**, версия с LoRA — **36,55%**. В этой паре адаптация снизила результат на **0,79 п.п.**

## Accuracy по категориям MMBench-ru

In [12]:
mmbench_categories = (
    mmbench.groupby(["model_name", "category"], as_index=False)
    .agg(accuracy=("is_correct", "mean"), examples=("is_correct", "size"))
    .sort_values(["model_name", "accuracy"])
)
mmbench_categories

,model_name,category,accuracy,examples
5,Qwen2.5-VL + LoRA #1,future_prediction,0.475000,120
18,Qwen2.5-VL + LoRA #1,spatial_relationship,0.482759,174
8,Qwen2.5-VL + LoRA #1,image_quality,0.500000,154
13,Qwen2.5-VL + LoRA #1,object_localization,0.596091,307
19,Qwen2.5-VL + LoRA #1,structuralized_imagetext_understanding,0.630872,149
...,...,...,...,...
102,SmolVLM базовая,attribute_recognition,0.427948,229
107,SmolVLM базовая,image_emotion,0.430000,200
109,SmolVLM базовая,image_scene,0.439698,398
114,SmolVLM базовая,ocr,0.489209,139


In [13]:
matrix = mmbench_categories.pivot(index="model_name", columns="category", values="accuracy")
fig = px.imshow(matrix, text_auto=".0%", aspect="auto", zmin=0, zmax=1, color_continuous_scale=[[0, "#fff7ed"], [0.5, "#fb923c"], [1, "#9a3412"]], labels={"color": "Accuracy"})
fig.update_xaxes(title="Категория", tickangle=-40)
fig.update_yaxes(title="")
style_plot(fig, "MMBench-ru: качество по категориям", "Accuracy для каждой модели и категории", 700).show()

In [14]:
category_total = (
    mmbench.groupby("category", as_index=False)
    .agg(accuracy=("is_correct", "mean"), examples=("is_correct", "size"))
    .sort_values("accuracy")
)
fig = px.bar(category_total, x="accuracy", y="category", orientation="h", text=category_total["accuracy"].map(lambda x: f"{x:.1%}"), color_discrete_sequence=[PINK], hover_data=["examples"])
fig.update_traces(textposition="outside", marker_line_color="#831843", marker_line_width=0.6)
fig.update_xaxes(range=[0, 0.9], tickformat=".0%", title="Средняя Accuracy")
fig.update_yaxes(title="")
style_plot(fig, "MMBench-ru: сложность категорий", "Среднее по всем моделям", 720).show()

### Вывод по категориям MMBench-ru

- В среднем по шести моделям лучше всего решаются `identity_reasoning` — **84,6%**, `image_scene` — **79,0%** и `celebrity_recognition` — **76,3%**.
- Самыми сложными оказались `spatial_relationship` — **41,6%**, `future_prediction` — **43,5%** и `image_quality` — **47,2%**.
- У Qwen LoRA #3 особенно высока Accuracy для `image_scene` — **97,7%**, `celebrity_recognition` — **96,0%** и `social_relation` — **94,8%**. Слабее всего эта версия справляется с пространственными отношениями — **49,4%**.
- Версии Qwen остаются заметно сильнее SmolVLM в большинстве категорий; основной резерв улучшения связан с пространственным пониманием и прогнозированием событий.

## Эффект LoRA

In [15]:
def metric_value(table, model_column, metric_column, model_name):
    values = table.loc[table[model_column].astype(str).eq(model_name), metric_column]
    if len(values) != 1:
        raise ValueError(f"Для модели {model_name!r} найдено значений: {len(values)}")
    return float(values.iloc[0])


gqa_base_qwen = metric_value(gqa_exact, "model", "exact_match", "Qwen2.5-VL базовая")
gqa_base_smol = metric_value(gqa_exact, "model", "exact_match", "SmolVLM базовая")
mmbench_base_qwen = metric_value(mmbench_metrics, "model_name", "accuracy", "Qwen2.5-VL базовая")
mmbench_base_smol = metric_value(mmbench_metrics, "model_name", "accuracy", "SmolVLM базовая")

lora_effect = pd.DataFrame([
    ["GQA-ru", "Qwen LoRA #1", metric_value(gqa_exact, "model", "exact_match", "Qwen2.5-VL + LoRA #1") - gqa_base_qwen],
    ["GQA-ru", "Qwen LoRA #2", metric_value(gqa_exact, "model", "exact_match", "Qwen2.5-VL + LoRA #2") - gqa_base_qwen],
    ["GQA-ru", "Qwen LoRA #3", metric_value(gqa_exact, "model", "exact_match", "Qwen2.5-VL + LoRA #3") - gqa_base_qwen],
    ["GQA-ru", "SmolVLM LoRA", metric_value(gqa_exact, "model", "exact_match", "SmolVLM + LoRA") - gqa_base_smol],
    ["MMBench-ru", "Qwen LoRA #1", metric_value(mmbench_metrics, "model_name", "accuracy", "Qwen2.5-VL + LoRA #1") - mmbench_base_qwen],
    ["MMBench-ru", "Qwen LoRA #2", metric_value(mmbench_metrics, "model_name", "accuracy", "Qwen2.5-VL + LoRA #2") - mmbench_base_qwen],
    ["MMBench-ru", "Qwen LoRA #3", metric_value(mmbench_metrics, "model_name", "accuracy", "Qwen2.5-VL + LoRA #3") - mmbench_base_qwen],
    ["MMBench-ru", "SmolVLM LoRA", metric_value(mmbench_metrics, "model_name", "accuracy", "SmolVLM + LoRA") - mmbench_base_smol],
], columns=["dataset", "model", "change"])
display(lora_effect)
fig = px.bar(lora_effect, x="model", y="change", color="dataset", barmode="group", text=lora_effect["change"].map(lambda x: f"{x:+.1%}"), color_discrete_map={"GQA-ru": BLUE, "MMBench-ru": GREEN})
fig.update_traces(textposition="outside")
fig.add_hline(y=0, line_color="#374151")
fig.update_yaxes(tickformat="+.0%", title="Изменение относительно базовой модели")
fig.update_xaxes(title="")
style_plot(fig, "Эффект LoRA", "Изменение основной метрики относительно базовой модели", 560).show()

,dataset,model,change
0,GQA-ru,Qwen LoRA #1,0.180747
1,GQA-ru,Qwen LoRA #2,0.217420
2,GQA-ru,Qwen LoRA #3,0.215373
3,GQA-ru,SmolVLM LoRA,0.088982
4,MMBench-ru,Qwen LoRA #1,-0.001279
5,MMBench-ru,Qwen LoRA #2,0.004092
6,MMBench-ru,Qwen LoRA #3,0.006905
7,MMBench-ru,SmolVLM LoRA,-0.007928


### Вывод по эффекту LoRA

На GQA-ru все адаптеры дали заметный прирост: Qwen LoRA #1 — **+18,1 п.п.**, LoRA #2 — **+21,7 п.п.**, LoRA #3 — **+21,5 п.п.**, SmolVLM LoRA — **+8,9 п.п.** На MMBench-ru изменения гораздо меньше: Qwen LoRA #1 получила **−0,1 п.п.**, LoRA #2 — **+0,4 п.п.**, LoRA #3 — **+0,7 п.п.**, SmolVLM LoRA — **−0,8 п.п.** Третья версия Qwen оказалась лучшей на MMBench-ru, но её преимущество над базовой моделью невелико.

## Итоги

In [16]:
gqa_results = gqa_exact.merge(gqa_bert, on="model")
gqa_results["dataset"] = "GQA-ru"
mmbench_results = mmbench_metrics.rename(columns={"model_name": "model"})
mmbench_results["dataset"] = "MMBench-ru"

gqa_results.to_csv(RESULT_DIR / "gqa_metrics.csv", index=False, encoding="utf-8-sig")
gqa_categories.to_csv(RESULT_DIR / "gqa_category_accuracy.csv", index=False, encoding="utf-8-sig")
mmbench_results.to_csv(RESULT_DIR / "mmbench_metrics.csv", index=False, encoding="utf-8-sig")
mmbench_categories.to_csv(RESULT_DIR / "mmbench_category_accuracy.csv", index=False, encoding="utf-8-sig")
gqa.to_csv(RESULT_DIR / "gqa_scored_predictions.csv", index=False, encoding="utf-8-sig")
mmbench.to_csv(RESULT_DIR / "mmbench_scored_predictions.csv", index=False, encoding="utf-8-sig")

display(gqa_results.sort_values("exact_match", ascending=False))
display(mmbench_results)

,model,exact_match,examples,bertscore_precision,bertscore_recall,bertscore_f1,dataset
0,Qwen2.5-VL + LoRA #2,0.537246,12216,0.908978,0.904446,0.906516,GQA-ru
1,Qwen2.5-VL + LoRA #3,0.535200,12216,0.907888,0.903537,0.905526,GQA-ru
2,Qwen2.5-VL + LoRA #1,0.500573,12216,0.900475,0.896211,0.898145,GQA-ru
3,Qwen2.5-VL базовая,0.319826,12216,0.810846,0.802432,0.806401,GQA-ru
4,SmolVLM + LoRA,0.291830,12216,0.853854,0.848809,0.851108,GQA-ru
5,SmolVLM базовая,0.202849,12216,0.689655,0.669268,0.679032,GQA-ru


,model,accuracy,examples,dataset
2,Qwen2.5-VL + LoRA #3,0.806138,3910,MMBench-ru
1,Qwen2.5-VL + LoRA #2,0.803325,3910,MMBench-ru
3,Qwen2.5-VL базовая,0.799233,3910,MMBench-ru
0,Qwen2.5-VL + LoRA #1,0.797954,3910,MMBench-ru
5,SmolVLM базовая,0.373402,3910,MMBench-ru
4,SmolVLM + LoRA,0.365473,3910,MMBench-ru


### Общий вывод

Проверка полноты подтвердила, что каждая модель оценена на всех доступных примерах: **12 216 вопросов GQA-ru** и **3 910 вопросов MMBench-ru**. Наборы ID совпадают с исходными датасетами, повторов и ошибок инференса нет.

Лучшей моделью для основной задачи проекта остаётся **`Qwen2.5-VL + LoRA #2`**. На полном GQA-ru она получила **53,72% Exact Match** и **0,907 BERTScore F1**. Qwen LoRA #3 находится совсем рядом — **53,52%** и **0,906** соответственно. Разница между ними составляет только **0,20 п.п. Exact Match**, но обе метрики немного выше у второй версии.

Qwen LoRA #1 получила **50,06% Exact Match** и **0,898 BERTScore F1**, а базовая Qwen — **31,98%** и **0,806**. Таким образом, все три адаптера заметно улучшают качество на GQA-ru; наибольший прирост Exact Match даёт LoRA #2 — **21,74 п.п.**

Категориальный анализ показывает, что LoRA #2 особенно сильна в вопросах об объектах — **83,1%** и атрибутах — **62,9%**. LoRA #3 немного лучше справляется с отношениями и категориями объектов, но её небольшого преимущества в этих разделах недостаточно, чтобы обойти LoRA #2 по общей метрике.

На полном MMBench-ru лидирует **Qwen LoRA #3 с 80,61% Accuracy**. Далее идут LoRA #2 с **80,33%**, базовая Qwen с **79,92%** и LoRA #1 с **79,80%**. Разрыв между версиями Qwen меньше одного процентного пункта, поэтому на этом датасете адаптеры меняют результат значительно слабее, чем на GQA-ru.

У SmolVLM LoRA повысила Exact Match на GQA-ru с **20,28%** до **29,18%** и BERTScore F1 с **0,679** до **0,851**. На MMBench-ru адаптированная версия получила **36,55%** против **37,34%** у базовой модели. В обоих датасетах семейство Qwen остаётся заметно сильнее SmolVLM.

Самыми сложными направлениями MMBench-ru остаются пространственные отношения и прогнозирование событий: средняя Accuracy для `spatial_relationship` равна **41,6%**, для `future_prediction` — **43,5%**. Даже у Qwen LoRA #3 пространственные отношения дают только **49,4%**, поэтому именно здесь остаётся самый заметный резерв улучшения.